# Tool calling from scratch

## Learning goals

- Understand that a model proposes tool calls but Python executes them
- Define tool descriptions and Pydantic argument schemas
- Build a registry with authorization and side-effect metadata
- Replace unsafe `eval` with an AST allowlist calculator
- Return structured observations for success and failure


## What tool calling really means

A model does not directly run Python functions. The application sends tool names, descriptions, and argument schemas to the model. The model may return a structured request such as `get_weather(city="Jaipur")`. The runtime then validates, authorizes, executes, and returns an observation.

This distinction is fundamental: model output is untrusted input to the tool runtime. A well-formed call can still be forbidden, expensive, repeated, or unsafe.


## Mental model

```mermaid
sequenceDiagram
  participant U as User
  participant M as Model
  participant R as Tool runtime
  participant T as Tool
  U->>M: Request plus available tool schemas
  M-->>R: Proposed tool name and arguments
  R->>R: Validate name, permissions, arguments, limits
  alt allowed and valid
    R->>T: Execute validated arguments
    T-->>R: Result or controlled error
    R-->>M: Structured observation
  else denied or invalid
    R-->>M: Safe error observation
  end
  M-->>U: Final answer or another proposed action
```

The runtime is the security and reliability boundary. It owns the registry, not the model. Unknown tool names must fail closed.


## Build a safe teaching calculator

Removing `__builtins__` from `eval` is not a sufficient general sandbox. Instead, parse the expression into an abstract syntax tree and evaluate only numeric constants plus explicitly allowed operators. Everything else is rejected.


In [ ]:
import ast
import operator


BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
UNARY_OPERATORS = {ast.UAdd: operator.pos, ast.USub: operator.neg}


def evaluate_numeric_node(node: ast.AST) -> int | float:
    if isinstance(node, ast.Expression):
        return evaluate_numeric_node(node.body)
    if isinstance(node, ast.Constant) and type(node.value) in {int, float}:
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in BINARY_OPERATORS:
        left = evaluate_numeric_node(node.left)
        right = evaluate_numeric_node(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large.")
        return BINARY_OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in UNARY_OPERATORS:
        return UNARY_OPERATORS[type(node.op)](evaluate_numeric_node(node.operand))
    raise ValueError(f"Unsupported expression element: {type(node).__name__}")


def calculate(expression: str) -> float:
    if len(expression) > 100:
        raise ValueError("Expression is too long.")
    tree = ast.parse(expression, mode="eval")
    result = float(evaluate_numeric_node(tree))
    if abs(result) > 1e12:
        raise ValueError("Result is outside the allowed range.")
    return result


print(calculate("(12 + 8) / 4"))
try:
    calculate("__import__('os').getcwd()")
except ValueError as error:
    print("Rejected unsafe expression:", error)


## Give every tool an argument schema

Descriptions help the model choose a tool. Schemas help the runtime reject malformed arguments. They should describe the narrowest interface the application is prepared to execute.


In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class ToolArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")


class CalculatorArgs(ToolArguments):
    expression: str = Field(min_length=1, max_length=100)


class TimeArgs(ToolArguments):
    timezone: str = Field(default="UTC", min_length=1, max_length=80)


class NotesArgs(ToolArguments):
    query: str = Field(min_length=2, max_length=100)


class WeatherArgs(ToolArguments):
    city: str = Field(min_length=2, max_length=100)


## Tool handlers are ordinary functions

Handlers should accept already validated arguments. This notebook keeps them deterministic and local: weather is explicitly a mock, notes are an in-memory corpus, and time uses the standard library's timezone database.


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError


LOCAL_NOTES = [
    "Jaipur: Amber Fort is outside the old city.",
    "Jaipur: reserve unhurried time around the City Palace area.",
    "Jodhpur: Mehrangarh Fort overlooks the blue city.",
]


def get_current_time(timezone: str = "UTC") -> dict[str, str]:
    current = datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "iso_time": current.isoformat()}


def search_local_notes(query: str) -> dict[str, object]:
    words = {word.lower() for word in query.split()}
    matches = [note for note in LOCAL_NOTES if words & set(note.lower().split())]
    return {"query": query, "matches": matches[:3]}


def get_weather(city: str) -> dict[str, str]:
    return {
        "city": city,
        "condition": "mock weather unavailable",
        "source": "offline teaching stub",
    }


## Registry: the runtime's source of truth

The registry binds a public name to its description, argument model, handler, and side-effect classification. Only registered tools can run. A write tool would normally require stronger authorization and explicit confirmation.


In [ ]:
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, Literal


@dataclass(frozen=True)
class ToolSpec:
    name: str
    description: str
    arguments_model: type[ToolArguments]
    handler: Callable[..., Any]
    side_effect: Literal["none", "read", "write"] = "none"


TOOLS = {
    "calculator": ToolSpec(
        name="calculator",
        description="Evaluate a basic numeric arithmetic expression.",
        arguments_model=CalculatorArgs,
        handler=calculate,
    ),
    "get_current_time": ToolSpec(
        name="get_current_time",
        description="Return the current time in an IANA timezone.",
        arguments_model=TimeArgs,
        handler=get_current_time,
        side_effect="read",
    ),
    "search_local_notes": ToolSpec(
        name="search_local_notes",
        description="Search the local travel-note collection.",
        arguments_model=NotesArgs,
        handler=search_local_notes,
        side_effect="read",
    ),
    "get_weather": ToolSpec(
        name="get_weather",
        description="Return offline mock weather for a city.",
        arguments_model=WeatherArgs,
        handler=get_weather,
        side_effect="read",
    ),
}


for tool in TOOLS.values():
    print(tool.name, tool.arguments_model.model_json_schema()["properties"])


## Validate, authorize, execute, observe

The dispatcher never indexes the registry before checking the name, and it never calls a handler with raw model arguments. Errors become bounded observations rather than uncaught stack traces or invented success messages.


In [ ]:
from pydantic import ValidationError


class ToolCall(BaseModel):
    call_id: str = Field(min_length=1)
    name: str = Field(min_length=1)
    arguments: dict[str, Any]


class ToolObservation(BaseModel):
    call_id: str
    ok: bool
    result: Any | None = None
    error: str | None = None


def execute_tool(call: ToolCall, allowed_tools: set[str]) -> ToolObservation:
    spec = TOOLS.get(call.name)
    if spec is None:
        return ToolObservation(call_id=call.call_id, ok=False, error="unknown_tool")
    if call.name not in allowed_tools:
        return ToolObservation(call_id=call.call_id, ok=False, error="tool_not_authorized")

    try:
        validated = spec.arguments_model.model_validate(call.arguments)
        result = spec.handler(**validated.model_dump())
        return ToolObservation(call_id=call.call_id, ok=True, result=result)
    except ValidationError as error:
        fields = [".".join(map(str, item["loc"])) for item in error.errors()]
        return ToolObservation(
            call_id=call.call_id,
            ok=False,
            error=f"invalid_arguments:{','.join(fields)}",
        )
    except (ArithmeticError, SyntaxError, ValueError, ZoneInfoNotFoundError) as error:
        return ToolObservation(
            call_id=call.call_id,
            ok=False,
            error=f"tool_error:{type(error).__name__}",
        )


In [ ]:
allowed = {"calculator", "get_current_time", "search_local_notes", "get_weather"}
test_calls = [
    ToolCall(call_id="call-1", name="calculator", arguments={"expression": "12 * 3"}),
    ToolCall(call_id="call-2", name="get_weather", arguments={"city": "Jaipur"}),
    ToolCall(call_id="call-3", name="get_weather", arguments={}),
    ToolCall(call_id="call-4", name="delete_everything", arguments={}),
]

for tool_call in test_calls:
    observation = execute_tool(tool_call, allowed)
    print(observation.model_dump(mode="json"))


## Production concerns beyond this notebook

- **Timeouts:** enforce them outside the handler and distinguish timeout from business failure.
- **Retries:** retry only idempotent operations or use idempotency keys.
- **Side effects:** require stronger authorization and user confirmation for writes.
- **Result size:** truncate or summarize oversized observations before returning them to the model.
- **Sensitive data:** redact logs and minimize data returned from tools.
- **Concurrency:** define whether calls can run in parallel and how partial failure works.
- **Auditability:** record call ID, tool name, authorization outcome, duration, and status—not hidden reasoning.


## Exercise

1. Add a divide-by-zero test and confirm it returns a controlled error observation.
2. Add a `write` tool that requires a separate confirmation flag; do not actually mutate external state.
3. Serialize the tool registry into model-facing names, descriptions, and JSON schemas.
4. Add duration measurement to `ToolObservation`.
5. Explain why validating arguments does not replace authorization.


## Summary

Tool calling is a protocol: the model proposes a structured action; the runtime checks the registry, authorization, schema, limits, and side effects; Python executes; and a structured observation returns to the model. Unknown and invalid calls fail closed. The model can request a capability, but it never owns permission to execute it.
